In [ ]:
# !pip install torch gpytorch botorch

## Define Objective Function

In [2]:
import torch
import numpy as np
import gpytorch

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [3]:
# Convert CMY volumes to RGB
# goal: achieve specific RGB values to optimize color accuracy
# input: CMY volumes, output: RGB values

# sample batch of target RGB values
target_rgb_batch = torch.tensor([
    [150.0, 100.0, 200.0],
    [100.0, 150.0, 50.0],
    [200.0, 200.0, 100.0]
], dtype=torch.float64)

# volume constraint (microliters)
max_volume = 270  

# feasibility constraint function
def feasibility_constraint(cmyk_volumes):
    return cmyk_volumes.sum(axis=1) - max_volume

# objective function
def objective_from_cmyk_batch(batch_size=10):
    num_targets = target_rgb_batch.size(0)  # number of target RGB values

    # generate batch samples of CMYK volumes using random.dirichlet
    proportions = np.random.dirichlet([1, 1, 1, 1], size=batch_size)  
    cmyk_volumes = proportions * max_volume  # Scale by max volume constraint
    V_c, V_m, V_y, V_k = cmyk_volumes[:, 0], cmyk_volumes[:, 1], cmyk_volumes[:, 2], cmyk_volumes[:, 3]

    # convert CMYK volumes to RGB values using the CMYK to RGB formula
    R = 255 * (1 - V_c / max_volume) * (1 - V_k / max_volume)
    G = 255 * (1 - V_m / max_volume) * (1 - V_k / max_volume)
    B = 255 * (1 - V_y / max_volume) * (1 - V_k / max_volume)
    generated_rgb = torch.tensor(np.column_stack([R, G, B]), dtype=torch.float64)

    # check if each sample in batch is feasible
    feasibility_mask = (feasibility_constraint(cmyk_volumes) <= 0).astype(float)
    feasibility_mask = torch.tensor(feasibility_mask, dtype=torch.float64) 

    # calculate error for each target RGB in batch
    errors = []
    for target_rgb in target_rgb_batch:
        error = torch.sqrt(torch.sum((generated_rgb - target_rgb) ** 2, dim=1))
        errors.append(error)

    # stack errors for all targets, then apply feasibility mask
    errors = torch.stack(errors, dim=1)  # shape: (batch_size, num_targets)
    weighted_errors = errors * feasibility_mask.unsqueeze(1)  # Apply feasibility mask
    mean_error = weighted_errors.mean()  # Overall mean error for feasible samples

    return mean_error, generated_rgb, cmyk_volumes, errors, feasibility_mask

## Set up Gaussian Process Model

### Generate Training Data

### Train GP Model on CMY-to-RGB Mapping

### Function to optimize for CMY values

## Optimize using BO

In [ ]:
"""
Considerations:
- hyperparameter tuning
- initial samples can impact convergence speed (try eg.Latin Hypercube Sampling)
- define stopping criteria (ie. max number of iterations, convergence of objective function,
    or reaching a certain performance threshold)
- color values contrained between 0 and 255 for RGB
"""

## Visualizing convergence of BO

In [ ]:
!pip install matplotlib

In [ ]:
"""
Exploitation:
Visualize how RGB values converge over iterations of the BO process. Basically, it tracks
the best RGB output obtained after each iteration of optimization.
"""

def plot_convergence(iterations, best_rgb_values):
    plt.figure(figsize=(12, 6))

    plt.subplot(3, 1, 1)
    plt.plot(iterations, best_rgb_values[:, 0], label='Red', color='red')
    plt.ylabel('Red Channel Value')
    plt.title('Convergence of RGB Values')
    plt.grid()
    plt.legend()

    plt.subplot(3, 1, 2)
    plt.plot(iterations, best_rgb_values[:, 1], label='Green', color='green')
    plt.ylabel('Green Channel Value')
    plt.grid()
    plt.legend()

    plt.subplot(3, 1, 3)
    plt.plot(iterations, best_rgb_values[:, 2], label='Blue', color='blue')
    plt.ylabel('Blue Channel Value')
    plt.xlabel('Iterations')
    plt.grid()
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_convergence(iterations, best_rgb_values)

In [ ]:
"""
Exploration:
Visualizes the predicted RGB values based on CMY values as predicted by the Gaussian
Process model. Basically, it helps in understanding the model's output across the 
entire input space.
"""

# Define a grid of CMY values for visualization
num_points = 100
cmy_grid = torch.linspace(0.0, 1.0, num_points)
cmy_mesh = torch.cartesian_prod(cmy_grid, cmy_grid, cmy_grid)

# Predict RGB values using the GP model
with torch.no_grad():
    gp.eval()
    pred_mean = gp.posterior(cmy_mesh).mean.detach()
    pred_var = gp.posterior(cmy_mesh).variance.detach()

# Visualize predictions
def plot_gp_predictions(cmy_mesh, pred_mean, pred_var):
    # Convert CMY to RGB for a scatter plot
    rgb_values = pred_mean.numpy()
    
    plt.figure(figsize=(10, 10))
    scatter = plt.scatter(rgb_values[:, 0], rgb_values[:, 1], c=rgb_values, s=10)
    plt.colorbar(scatter, label='RGB Value')
    plt.xlabel('Red Channel Value')
    plt.ylabel('Green Channel Value')
    plt.title('GP Predictions in RGB Space')
    plt.grid()
    plt.show()

plot_gp_predictions(cmy_mesh, pred_mean, pred_var)